In [ ]:
import vitaldb
import pandas as pd
import numpy as np
from scipy import signal
import ssl
import certifi

# SSL setup for vitaldb connection
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = lambda: ssl.create_default_context(cafile=certifi.where())


def get_eligible_patients(clinical_df, min_age=60, max_asa=2):
    """Filters patients based on age and ASA criteria."""
    clinical_df['age'] = pd.to_numeric(clinical_df['age'], errors='coerce')
    clinical_df['asa'] = pd.to_numeric(clinical_df['asa'], errors='coerce')
    
    filtered = clinical_df[
        (clinical_df['age'] > min_age) & 
        (clinical_df['asa'] <= max_asa)
    ]
    
    eligible_ids = filtered.index.tolist()
    
    print(f"Found {len(eligible_ids)} patients matching criteria:")
    print(f"   - Age > {min_age}")
    print(f"   - ASA ≤ {max_asa}")
    
    return eligible_ids


def detect_r_peaks(ecg_signal, sampling_rate=500):
    """Detects R-peaks in an ECG signal."""
    # Bandpass filter
    nyquist = sampling_rate / 2
    low = 0.5 / nyquist
    high = 40 / nyquist
    b, a = signal.butter(4, [low, high], btype='band')
    filtered_ecg = signal.filtfilt(b, a, ecg_signal)
    
    # Square the signal
    squared = filtered_ecg ** 2
    
    # Find peaks
    peak_distance = int(0.3 * sampling_rate)
    threshold = np.mean(squared) + 0.5 * np.std(squared)
    peaks, _ = signal.find_peaks(squared, distance=peak_distance, height=threshold)
    
    return peaks


def calculate_hrv_features(rr_intervals):
    """Calculates HRV features from R-R intervals."""
    if len(rr_intervals) < 5:
        return {'mean_hr': np.nan, 'sdnn': np.nan, 'rmssd': np.nan}
    
    sdnn = np.std(rr_intervals)
    successive_diffs = np.diff(rr_intervals)
    rmssd = np.sqrt(np.mean(successive_diffs ** 2))
    mean_hr = 60 / np.mean(rr_intervals)
    
    return {'mean_hr': mean_hr, 'sdnn': sdnn, 'rmssd': rmssd}


def extract_ecg_features(ecg_waveform, sampling_rate=500, window_seconds=30):
    """Extracts HRV features from ECG waveform in sliding windows."""
    window_samples = window_seconds * sampling_rate
    num_windows = len(ecg_waveform) // window_samples
    
    hrv_features = []
    
    for i in range(num_windows):
        start = i * window_samples
        end = start + window_samples
        window = ecg_waveform[start:end]
        
        peaks = detect_r_peaks(window, sampling_rate)
        
        if len(peaks) > 1:
            rr_intervals = np.diff(peaks) / sampling_rate
            features = calculate_hrv_features(rr_intervals)
        else:
            features = {'mean_hr': np.nan, 'sdnn': np.nan, 'rmssd': np.nan}
        
        features['time_seconds'] = (start + end) / 2 / sampling_rate
        hrv_features.append(features)
    
    return pd.DataFrame(hrv_features)


def classify_multimodal_risk(vitals_df, hrv_df, patient_info):
    """
    Classifies patient risk by combining vital signs with ECG-derived HRV features.
    """
    # Merge HRV features with vital signs
    vitals_df = vitals_df.copy()
    vitals_df['time_seconds'] = vitals_df.index.total_seconds()
    
    merged = pd.merge_asof(
        vitals_df.sort_values('time_seconds'),
        hrv_df.sort_values('time_seconds'),
        on='time_seconds',
        direction='nearest',
        tolerance=30
    )
    merged.index = vitals_df.index
    
    risk_scores = pd.Series(0.0, index=merged.index)
    
    
    # 1. VITAL SIGNS
    # Use the track names as column names
    hr = merged['Solar8000/HR']
    spo2 = merged['Solar8000/PLETH_SPO2']
    bt = merged['Solar8000/BT']
    
    risk_scores += (hr < 40).astype(int) * 3
    risk_scores += (hr > 130).astype(int) * 3
    risk_scores += ((hr >= 40) & (hr < 45)).astype(int) * 1
    risk_scores += ((hr > 120) & (hr <= 130)).astype(int) * 1
    
    risk_scores += (spo2 < 88).astype(int) * 4
    risk_scores += ((spo2 >= 88) & (spo2 < 92)).astype(int) * 2
    
    risk_scores += (bt < 35).astype(int) * 2
    risk_scores += (bt > 39).astype(int) * 2
    
    # 2. HRV FEATURES
    sdnn = merged['sdnn']
    rmssd = merged['rmssd']
    
    risk_scores += (sdnn < 0.015).astype(int) * 2
    risk_scores += (rmssd < 0.010).astype(int) * 1
    
    # 3. CLASSIFICATION THRESHOLDS
    status = pd.Series('healthy', index=merged.index)
    status[risk_scores >= 4] = 'dangerous'
    status[(risk_scores >= 2) & (risk_scores < 4)] = 'risky'
    
    return status

def clean_raw_vitals(df_vitals, vital_tracks):
    """Performs initial, essential cleaning on raw vital sign tracks."""

    df_clean = df_vitals.copy()
    initial_count = len(df_clean)
    
    # Map track names to cleaner variables
    hr_track = vital_tracks[0] # Assuming HR is first
    bt_track = vital_tracks[1] # Assuming BT is second
    spo2_track = vital_tracks[2] # Assuming SPO2 is third
    
    # 1. REMOVE NULL/NaN VALUES (Essential first step)
    df_clean.dropna(inplace=True)
    
    if df_clean.empty:
        return df_clean

    # 2. REMOVE PHYSIOLOGICALLY IMPOSSIBLE VALUES (Hard limits)
    impossible_mask = pd.Series(False, index=df_clean.index)
    
    # Heart Rate: Human limits are ~20-250 bpm
    impossible_mask |= (df_clean[hr_track] < 20) | (df_clean[hr_track] > 250)
    
    # SpO2: Must be between 50-100% (lower are usually errors/disconnects)
    impossible_mask |= (df_clean[spo2_track] < 50) | (df_clean[spo2_track] > 100)
    
    # Temperature: Human survival range is ~28-44°C
    impossible_mask |= (df_clean[bt_track] < 28) | (df_clean[bt_track] > 44)
    
    df_clean = df_clean[~impossible_mask]

    # 3. REMOVE FLAT/CONSTANT MEASUREMENTS (Sensor errors)
    flat_mask = pd.Series(False, index=df_clean.index)
    
    for col in vital_tracks:
        # Standard deviation of 0 in a window of 5 means all values are identical
        rolling_std = df_clean[col].rolling(window=5, center=True, min_periods=1).std()
        flat_mask |= (rolling_std == 0)
    
    df_clean = df_clean[~flat_mask]

    # 4. REMOVE SUDDEN JUMPS (Measurement artifacts)
    jump_mask = pd.Series(False, index=df_clean.index)
    
    max_changes = {
        hr_track: 30,       # Max 30 bpm jump
        spo2_track: 10,     # Max 10% jump
        bt_track: 1.5       # Max 1.5°C jump
    }
    
    for col, max_change in max_changes.items():
        diff = df_clean[col].diff().abs()
        jump_mask |= (diff > max_change)
    
    df_clean = df_clean[~jump_mask]
    
    # Final cleanup of all NaNs generated by rolling/diff operations
    df_clean.dropna(inplace=True)
    
    removed_count = initial_count - len(df_clean)
    if removed_count > 0:
         print(f"      - Raw cleaning removed {removed_count} artifacts.")
         
    return df_clean


def process_patient_with_ecg(case_id, vital_tracks, ecg_track, patient_info, 
                             interval=10, sample_size=None):
    """
    Loads and processes both vital signs and ECG for a single patient,
    running essential preprocessing *before* classification.
    """
    try:
        # Load vital signs
        df_vitals = vitaldb.load_case(case_id, vital_tracks, interval=interval)
        
        if isinstance(df_vitals, np.ndarray):
            time_index = pd.to_timedelta(np.arange(len(df_vitals)) * interval, unit='s')
            df_vitals = pd.DataFrame(df_vitals, columns=vital_tracks, index=time_index)
        
        if df_vitals.empty:
            return None
        
        # --- Preprocessing Step (Runs BEFORE Classification) ---
        df_vitals_clean = clean_raw_vitals(df_vitals, vital_tracks)
        
        if df_vitals_clean.empty:
            return None

        # Load ECG waveform (this is high frequency, so cleaning isn't applied here)
        ecg_data = vitaldb.load_case(case_id, ecg_track, interval=1/500)
        
        if isinstance(ecg_data, np.ndarray):
            ecg_waveform = ecg_data.flatten()
        else:
            ecg_waveform = ecg_data[ecg_track[0]].values
        
        # Extract HRV features from ECG
        hrv_df = extract_ecg_features(ecg_waveform, sampling_rate=500, window_seconds=30)
        
        if len(hrv_df) == 0:
            return None
        
        # Apply multimodal risk classification 
        df_vitals_clean['status'] = classify_multimodal_risk(df_vitals_clean, hrv_df, patient_info)
        
        # Create clean output DataFrame with renamed columns
        df_clean = pd.DataFrame()
        df_clean['temperature'] = df_vitals_clean['Solar8000/BT'].values
        df_clean['heath_rate'] = df_vitals_clean['Solar8000/HR'].values
        df_clean['blood_oxygen'] = df_vitals_clean['Solar8000/PLETH_SPO2'].values
        df_clean['status'] = df_vitals_clean['status'].values
        
        # Sample if requested
        if sample_size is not None and len(df_clean) > sample_size:
            df_clean = df_clean.sample(n=sample_size, random_state=42)
        
        return df_clean
        
    except Exception as e:
        print(f"   Error processing case {case_id}: {str(e)[:100]}")
        return None

def build_ecg_enhanced_dataset(case_ids, clinical_df, vital_tracks, ecg_track,
                               samples_per_patient=50, max_patients=None):
    """
    Builds dataset using ECG-enhanced multimodal risk classification.
    
    Args:
        case_ids (list): List of eligible patient IDs
        clinical_df (pd.DataFrame): Clinical data for all patients
        vital_tracks (list): Vital sign tracks to load
        ecg_track (list): ECG track to load
        samples_per_patient (int): Number of samples per patient
        max_patients (int): Maximum number of patients (None = all)
        
    Returns:
        pd.DataFrame: Combined dataset with ECG-based risk classification
    """
    all_data = []
    patients_with_data = 0
    
    if max_patients is not None:
        case_ids = case_ids[:max_patients]
    
    print(f"\nProcessing {len(case_ids)} eligible patients with ECG analysis...")
    print(f"Extracting {samples_per_patient} samples per patient...\n")
    
    for i, case_id in enumerate(case_ids, 1):
        if i % 5 == 0:
            print(f"  Processed {i}/{len(case_ids)} patients "
                  f"({patients_with_data} with valid data)...")
        
        # Get patient info for this case
        try:
            patient_info = clinical_df.loc[case_id].to_dict()
        except KeyError:
            continue
        
        # Process patient with ECG
        patient_data = process_patient_with_ecg(
            case_id, vital_tracks, ecg_track, patient_info,
            interval=10, sample_size=samples_per_patient
        )
        
        if patient_data is not None and len(patient_data) > 0:
            all_data.append(patient_data)
            patients_with_data += 1
    
    if len(all_data) == 0:
        raise ValueError("No valid data found for any patient!")
    
    final_df = pd.concat(all_data, ignore_index=True)
    
    print(f"\n=== Dataset Creation Complete ===")
    print(f"Patients with valid  {patients_with_data}")
    print(f"Total samples: {len(final_df)}")
    print(f"\nStatus distribution:")
    print(final_df['status'].value_counts())
    
    return final_df


In [ ]:
# --- CONFIGURATION ---
MIN_AGE = 60
MAX_ASA = 2
SAMPLES_PER_PATIENT = 10
MAX_PATIENTS = None  # Set to None to process all eligible patients 
VITAL_TRACKS = ['Solar8000/HR', 'Solar8000/BT', 'Solar8000/PLETH_SPO2']
ECG_TRACK = ['SNUADC/ECG_II']
OUTPUT_FILE = 'dataset_enhanced.csv'
# --- END CONFIGURATION ---

try:
    # Step 1: Load and filter patients
    print("=== Step 1: Loading and Filtering Patients ===")
    # Assuming 'clinical_data.csv' exists for vitaldb context
    clinical_df = pd.read_csv('clinical_data.csv', index_col='caseid')
    eligible_patients = get_eligible_patients(clinical_df, 
                                             min_age=MIN_AGE, 
                                             max_asa=MAX_ASA)
    
    if len(eligible_patients) == 0:
        raise ValueError("No patients match the specified criteria!")
    
    # Step 2: Build dataset with ECG-enhanced classification and integrated PREPROCESSING
    print("\n=== Step 2: Building ECG-Enhanced Dataset with Preprocessing ===")
    final_dataset = build_ecg_enhanced_dataset(
        eligible_patients,
        clinical_df,
        VITAL_TRACKS,
        ECG_TRACK,
        samples_per_patient=SAMPLES_PER_PATIENT,
        max_patients=MAX_PATIENTS
    )
    
    # Step 3: Display sample data
    print("\n=== Step 3: Dataset Preview ===")
    print("\nFirst 10 rows:")
    print(final_dataset.head(10))
    
    print("\nStatistical summary:")
    print(final_dataset.describe())
    
    # Step 4: Save to CSV
    print(f"\n=== Step 4: Saving Dataset ===")
    final_dataset.to_csv(OUTPUT_FILE, index=False)
    print(f"Dataset saved to: {OUTPUT_FILE}")
    
    # Class balance check
    print("\n=== Class Balance ===")
    status_counts = final_dataset['status'].value_counts()
    total = len(final_dataset)
    for status, count in status_counts.items():
        percentage = (count / total) * 100
        print(f"{status}: {count} ({percentage:.1f}%)")
    
    print("\n=== SUCCESS ===")
    print(f"ECG-enhanced dataset ready with {len(final_dataset)} samples!")
    
except Exception as e:
    print(f"\nAn error occurred: {e}")
    import traceback
    traceback.print_exc()



=== Step 1: Loading and Filtering Patients ===
Found 2417 patients matching criteria:
   - Age > 60
   - ASA ≤ 2

=== Step 2: Building ECG-Enhanced Dataset with Preprocessing ===

Processing 2417 eligible patients with ECG analysis...
Extracting 10 samples per patient...

      - Raw cleaning removed 1092 artifacts.
      - Raw cleaning removed 433 artifacts.
      - Raw cleaning removed 1994 artifacts.
      - Raw cleaning removed 483 artifacts.
  Processed 5/2417 patients (4 with valid data)...
      - Raw cleaning removed 555 artifacts.
      - Raw cleaning removed 1059 artifacts.
      - Raw cleaning removed 1977 artifacts.
      - Raw cleaning removed 438 artifacts.
      - Raw cleaning removed 2700 artifacts.
  Processed 10/2417 patients (8 with valid data)...
      - Raw cleaning removed 2569 artifacts.
      - Raw cleaning removed 1211 artifacts.
      - Raw cleaning removed 279 artifacts.
      - Raw cleaning removed 1451 artifacts.
      - Raw cleaning removed 1053 artifacts.

In [16]:
final_dataset.to_csv("final_dataset.csv", index=False)

In [1]:
def balance_dataset_ratio_based(df, target_ratios, target_column='status', random_state=42):
    """
    Balances the dataset by undersampling majority classes to achieve a 
    specific target ratio, based on the size of the minority class.
    
    Args:
        df (pd.DataFrame): The input dataset.
        target_ratios (dict): Dictionary of {status: ratio (0.0 to 1.0)}.
        target_column (str): The name of the column to balance.
        random_state (int): Seed for reproducibility.
        
    Returns:
        pd.DataFrame: The balanced dataset.
    """
    print("\n--- Starting Ratio-Based Undersampling for Class Balancing ---")
    
    status_counts = df[target_column].value_counts()
    
    # 1. Identify the minority class and its count based on the lowest target ratio
    # Assuming 'dangerous' is the minority class (smallest count) and has a target ratio > 0
    minority_class = 'dangerous'
    if minority_class not in status_counts or target_ratios.get(minority_class, 0) == 0:
        raise ValueError("Minority class 'dangerous' not found or its target ratio is zero.")
        
    minority_count = status_counts[minority_class]
    minority_ratio = target_ratios[minority_class]
    
    # 2. Calculate the total size of the new dataset
    N_final = int(minority_count / minority_ratio)
    print(f"Minority Class: '{minority_class}' with {minority_count} samples ({minority_ratio*100:.0f}% of target total).")
    print(f"Target total dataset size (N_final): {N_final}")

    df_list = []
    
    # 3. Process and sample each class
    for status, target_ratio in target_ratios.items():
        df_status = df[df[target_column] == status]
        original_count = len(df_status)
        
        # Calculate the required count for this class
        required_count = int(N_final * target_ratio)
        
        if original_count < required_count:
            # This should only happen for the minority class, but print a warning
            print(f" ! WARNING: Class '{status}' requires {required_count}, but only has {original_count}. Using all available samples.")
            sample_count = original_count
            df_list.append(df_status)
        else:
            # Undersample the current status group
            sample_count = required_count
            df_undersampled = df_status.sample(
                n=sample_count, 
                random_state=random_state
            )
            df_list.append(df_undersampled)
        
        print(f" - Class '{status}' sampled: {original_count} -> {sample_count} samples ({target_ratio*100:.0f}% of total)")

    # 4. Combine the balanced classes
    df_balanced = pd.concat(df_list).sample(frac=1, random_state=random_state).reset_index(drop=True)
    
    print("--- Ratio-Based Undersampling Complete ---")
    print(f"Balanced Dataset Size: {len(df_balanced)} samples.")
    print("New Status Distribution:")
    print(df_balanced[target_column].value_counts())
    
    return df_balanced

In [ ]:
final_dataset = pd.read_csv("final_dataset.csv")
TARGET_RATIOS = {
        'dangerous': 0.05,  # 5%
        'risky': 0.15,      # 15%
        'healthy': 0.80     # 80%
    }
    
balanced_dataset = balance_dataset_ratio_based(
        final_dataset, 
        target_ratios=TARGET_RATIOS
    )


--- Starting Ratio-Based Undersampling for Class Balancing ---
Minority Class: 'dangerous' with 473 samples (5% of target total).
Target total dataset size (N_final): 9460
 - Class 'dangerous' sampled: 473 -> 473 samples (5% of total)
 - Class 'risky' sampled: 4541 -> 1419 samples (15% of total)
 - Class 'healthy' sampled: 10088 -> 7568 samples (80% of total)
--- Ratio-Based Undersampling Complete ---
Balanced Dataset Size: 9460 samples.
New Status Distribution:
status
healthy      7568
risky        1419
dangerous     473
Name: count, dtype: int64


In [7]:
balanced_dataset

,temperature,hr,spo2,status
0,36.700001,60.0,94.0,healthy
1,35.100002,65.0,99.0,healthy
2,36.200001,74.0,100.0,healthy
3,35.200001,50.0,98.0,healthy
4,35.799999,68.0,99.0,healthy
...,...,...,...,...
9455,35.100002,68.0,98.0,healthy
9456,35.700001,75.0,100.0,healthy
9457,35.799999,92.0,99.0,healthy
9458,36.000000,82.0,100.0,risky


In [8]:
balanced_dataset.to_csv("realistic_elderly_data.csv", index=False)